In [1]:
import numpy as np
import polars as pl
import typol as tp
from matplotlib import pyplot as plt
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import ConfusionMatrixDisplay, confusion_matrix
from sklearn.model_selection import train_test_split
from catboost import CatBoostClassifier, Pool

from Answer import Answer, AnswerQ11NullsAsOnes, PredictedAndActualQ11, AnswerParts1_6_NoQ1, PredictedAndActualQ1, AnswerParts1_6, AnswerParts1_2_3_4_5_6_NotQ11, OnlyQ11

In [2]:
labels = {
    1: "None",
    2: "1...2",
    3: "3...4",
    4: "5...6",
    5: "6...10",
    6: "10...20",
    7: ">20",
}

In [3]:
# TODO Predict a person's reading habits (how much/often they read) according to their answers
def predict_habits(data: tp.DataFrame[AnswerQ11NullsAsOnes]) -> tp.DataFrame[PredictedAndActualQ11]:
    # we're trying to predict the enum HowManyBooksReadInTheLastSixMonths aka Question 11
    training_data: tp.DataFrame[AnswerWithoutQ11] = data.transform(AnswerWithoutQ11)
    X_train, X_test, y_train, y_test = train_test_split(
        training_data.dataframe, data[Answer.Q11_1].to_list(), test_size=0.20, random_state=0
    )
    X_train = tp.DataFrame(AnswerWithoutQ11, X_train)
    # it wouldn't do to just train on X_train and y_train because the model would correctly,
    # but not very usefully, learn that empty answers => non-reader
    books_read_predictor: RandomForestClassifier = RandomForestClassifier()
    books_read_predictor.fit(X_train, y_train)
    predictions: np.ndarray[tuple[int], np.dtype[np.int32]] = books_read_predictor.predict(X_test)
    as_dict = {
        "Predicted": predictions,
        "Actual": y_test,
    }
    results = tp.DataFrame(PredictedAndActualQ11, as_dict)
    return results


In [42]:
def binary_is_reader(x: int) -> int:
        if x == 1:
            return 0
        else:
            return 1

In [43]:
def predict_is_reader_parts_1_6(data: tp.DataFrame[AnswerParts1_6]) -> tp.DataFrame[PredictedAndActualQ1]:
    # we're trying to predict the enum HowManyBooksReadInTheLastSixMonths aka Question 11
    data_without_Q1 = data.transform(AnswerParts1_6_NoQ1)

    data_with_binary_Q1 = data.with_columns(
        data.s.Q1.map(binary_is_reader)
    )
    only_binary_Q1 = data_with_binary_Q1[data_with_binary_Q1.s.Q1]

    X_train, X_test, y_train, y_test = train_test_split(
        data_without_Q1.dataframe,
        only_binary_Q1.to_list(),
        test_size=0.20,
        random_state=0
    )
    # we need to predict whether a person considers themself a reader.
    # we can only predict this based on the answers they give in parts 1 and 6,
    # because they're the only parts that non-readers and readers both fill in

    # is_reader_predictor = RandomForestClassifier(
    #     n_estimators=100, # 209 features
    #     random_state=0,
    # )
    is_reader_predictor = CatBoostClassifier(
        verbose=True,
        random_state=0,
    )
    is_reader_predictor.fit(X_train, y_train)
    # tree = is_reader_predictor.plot_tree(0)
    # tree.render()
    predictions: np.ndarray[tuple[int], np.dtype[np.int32]] = is_reader_predictor.predict(X_test)
    as_dict = {
        "Predicted": predictions,
        "Actual": y_test,
    }
    results = tp.DataFrame(PredictedAndActualQ1, as_dict)
    return results

In [44]:
def find_best_params_for_catboost(data: tp.DataFrame[AnswerParts1_6]) -> dict[str, dict[str, str]]:
    grid = {
        "learning_rate": [0.03, 0.1],
        "depth": [4, 6, 10],
        "l2_leaf_reg": [1, 3, 5, 7, 9],
        "num_trees": [100, 500, 1000],
        # "loss_function": ["Logloss"],
        # "eval_metric": ["Logloss", "BalancedAccuracy", "F1", "MCC"],
        "auto_class_weights": ["None", "Balanced", "SqrtBalanced"]
    }
    is_reader_predictor = CatBoostClassifier(
        verbose=True,
        random_state=0,
    )
    search_results: dict[str, dict[str, str]] = is_reader_predictor.grid_search(
        X=data.transform(AnswerParts1_6_NoQ1).dataframe,
        y=data.with_columns(
            data.s.Q1.map(binary_is_reader)
        )[data.s.Q1].to_list(),
        verbose=True,
        partition_random_seed=0,
        param_grid=grid
    )
    print(search_results)
    return search_results

In [45]:
def visualize_is_reader_predictions(data: tp.DataFrame[PredictedAndActualQ1]):
    actual = data[data.s.Actual].to_list()
    predicted = data[data.s.Predicted].to_list()

    cm = confusion_matrix(actual, predicted, labels=[1, 0])
    correct = np.eye(2, dtype=bool)

    fig, ax = plt.subplots()

    ax.imshow(np.ma.masked_where(~correct, cm), cmap="Greens")
    ax.imshow(np.ma.masked_where(correct, cm), cmap="Reds")

    for column in range(2):
        for row in range(2):
            vmax = cm[correct].max() if correct[column, row] else cm[~correct].max()
            ax.text(
                row, column, cm[column, row],
                ha="center",
                va="center",
                weight="bold",
                color="white" if (cm[column, row] / vmax) > 0.5 else "black",
            )

    ax.set_xticks([0, 1], ["Reader", "Non-reader"])
    ax.set_yticks([0, 1], ["Reader", "Non-reader"])
    ax.set_xlabel("Predicted")
    ax.set_ylabel("Actual")
    ax.set_title("Predicted vs. Actual")
    fig.tight_layout()
    plt.show()

In [46]:
def visualize_readership(data: tp.DataFrame[AnswerQ11NullsAsOnes]):
    df = data.dataframe.clone()
    counts = (
        df["Q11_1"]
        .value_counts()
        .sort("Q11_1")
    )

    fig, ax = plt.subplots()
    ax.bar(
        x=counts["Q11_1"].to_list(),
        height=counts["count"].to_list(),
    )

    ax.set_xlabel("How many books have you read in the last six months?")
    ax.set_xticks(list(labels))
    ax.set_xticklabels(list(labels.values()), rotation=45, ha="center")
    ax.set_ylabel("Count")

In [47]:
def visualize_predictions(results: tp.DataFrame[PredictedAndActualQ11]):
    df = results.dataframe
    actual = df["Actual"].to_numpy()
    predicted = df["Predicted"].to_numpy()

    fig, ax = plt.subplots(figsize=(9, 7))
    ConfusionMatrixDisplay.from_predictions(
        actual,
        predicted,
        labels=list(labels),
        display_labels=list(labels.values()),
        cmap="Blues",
        values_format="d",
        xticks_rotation=45,
        ax=ax,
    )
    ax.set_title("Predicted vs. Actual amount read")
    fig.tight_layout()
    plt.show()


In [49]:
# TODO Assess the factors most responsible for determining a person's reading habits (PCA)
# TODO Determine the feasibility of a model for recommending books to a person according to their answers

def main():
    raw_data: pl.DataFrame = pl.read_csv("Lugeja25.tab", has_header=True, separator="\t")
    typed_data: tp.DataFrame[Answer] = tp.DataFrame(Answer, values=raw_data)
    # print(typed_data.head(1))
    # predict_habits(typed_data)
    q11_nulls_to_zero_data: tp.DataFrame[AnswerQ11NullsAsOnes] = typed_data.with_columns(typed_data.s.Q11_1.fill_null(strategy="one")).transform(AnswerQ11NullsAsOnes)
    # visualize_readership(data=q11_nulls_to_zero_data)

    is_reader_data = typed_data.transform(AnswerParts1_6)
    find_best_params_for_catboost(is_reader_data)
    is_reader_predictions: tp.DataFrame[PredictedAndActualQ1] = predict_is_reader_parts_1_6(is_reader_data)
    visualize_is_reader_predictions(is_reader_predictions)

    #v = predict_habits(data=q11_nulls_to_zero_data)
    #print(v)
    # visualize_predictions(v)

if __name__ == "__main__":
    main()

0:	learn: 0.6574960	test: 0.6560973	best: 0.6560973 (0)	total: 6.58ms	remaining: 652ms
1:	learn: 0.5806431	test: 0.5764397	best: 0.5764397 (1)	total: 13.6ms	remaining: 665ms
2:	learn: 0.5193711	test: 0.5130492	best: 0.5130492 (2)	total: 19.3ms	remaining: 625ms
3:	learn: 0.4803146	test: 0.4731477	best: 0.4731477 (3)	total: 24.3ms	remaining: 582ms
4:	learn: 0.4584116	test: 0.4501740	best: 0.4501740 (4)	total: 28.8ms	remaining: 547ms
5:	learn: 0.4252024	test: 0.4151972	best: 0.4151972 (5)	total: 33.6ms	remaining: 527ms
6:	learn: 0.3840885	test: 0.3721528	best: 0.3721528 (6)	total: 38.5ms	remaining: 512ms
7:	learn: 0.3587442	test: 0.3461533	best: 0.3461533 (7)	total: 43.3ms	remaining: 497ms
8:	learn: 0.3444754	test: 0.3306275	best: 0.3306275 (8)	total: 48.6ms	remaining: 491ms
9:	learn: 0.3319372	test: 0.3170111	best: 0.3170111 (9)	total: 53.9ms	remaining: 485ms
10:	learn: 0.3197178	test: 0.3036689	best: 0.3036689 (10)	total: 58.5ms	remaining: 473ms
11:	learn: 0.2892174	test: 0.2721074	best

KeyboardInterrupt: 